### Imports

In [33]:
# Installs (Colab/runtime)
%pip -q install langchain langchain-text-splitters langchain-community bs4 sentence-transformers
%pip -q install -U "langchain[google-genai]"
%pip -q install -U "langchain-core"

import getpass
import os
import re
import shutil
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import CSVLoader
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore

### Google Gemini

In [34]:
os.environ["GOOGLE_API_KEY"] =""
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

model = init_chat_model("google_genai:gemini-2.5-flash")

### Embeddings

In [35]:
# Embeddings: default to LOCAL to avoid Gemini free-tier embed quota.
USE_GEMINI_EMBEDDINGS = False

if USE_GEMINI_EMBEDDINGS:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
else:
    from langchain_community.embeddings import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = InMemoryVectorStore(embeddings)

### Data and Chunks

In [36]:
csv_path = Path("fashion.csv")

loader = CSVLoader(file_path=str(csv_path), encoding="utf-8")
docs = loader.load()

all_splits = docs

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

### RAG Agent

In [40]:
# --- RAG chain via dynamic prompt (middleware) ---
# This runs retrieval automatically on every user message and injects the
# retrieved content into the model prompt.
@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    last_msg = request.state["messages"][-1]

    # Be robust across different message object shapes
    last_query = getattr(last_msg, "text", None) or getattr(last_msg, "content", "")

    retrieved_docs = vector_store.similarity_search(last_query, k=3)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful shopping assistant for a fashion catalog. Use the following product catalog context to answer. "
        "If the answer isn't in the context, say you don't know.\n\n"
        "Output format (IMPORTANT):\n"
        "- Return your final answer as GitHub-flavored Markdown.\n"
        "- When you list products, for each product include: ProductTitle, ProductId, Size (if present), and ImageURL.\n"
        "- Also embed the image preview using Markdown image syntax on its own line: ![ProductTitle](ImageURL)\n\n"
        "CATALOG CONTEXT:\n"
        f"{docs_content}"
    )

    return system_message

agent = create_agent(model, tools=[], middleware=[prompt_with_context])

### Query

In [45]:
query = "hi, do you have girls pink top?"

resp = agent.invoke({"messages": [{"role": "user", "content": query}]})
md = resp["messages"][-1].content

display(Markdown(md))

Yes, I do! Here are a few options for girls' pink tops:

*   **Doodle Kids Girls Pink I love Shopping Top**
    ProductId: 23623
    Size: Large, X-Small, X-Large, Medium, Small
    ImageURL: http://assets.myntassets.com/v1/images/style/properties/ef9685293a987f515492addd034006bf_images.jpg
    ![Doodle Kids Girls Pink I love Shopping Top](http://assets.myntassets.com/v1/images/style/properties/ef9685293a987f515492addd034006bf_images.jpg)

*   **Little Miss Girls Naughty Pink Top**
    ProductId: 36744
    Size: X-Small
    ImageURL: http://assets.myntassets.com/v1/images/style/properties/076a9c14bd34ba5cd78012159e7d78fb_images.jpg
    ![Little Miss Girls Naughty Pink Top](http://assets.myntassets.com/v1/images/style/properties/076a9c14bd34ba5cd78012159e7d78fb_images.jpg)

*   **Doodle Girls Pink Top**
    ProductId: 41756
    Size: Medium, X-Small, Small, X-Large
    ImageURL: http://assets.myntassets.com/v1/images/style/properties/b0d1e113330843fc949c25eebefcf706_images.jpg
    ![Doodle Girls Pink Top](http://assets.myntassets.com/v1/images/style/properties/b0d1e113330843fc949c25eebefcf706_images.jpg)